# Google Gemini 모델(ChatGoogleGenerativeAI) 실습

지금까지는 OpenAI(`ChatOpenAI`) 모델만 써봤는데, 이번엔 **Google Gemini** 모델을 LangChain으로 호출하는 방법을 실습했다. LangChain은 모델 제공사가 달라도 거의 동일한 인터페이스(`invoke`, `stream`, 프롬프트/체인 연결 방식 등)로 쓸 수 있게 감싸주기 때문에, `ChatOpenAI` 대신 `ChatGoogleGenerativeAI`로 바꿔도 사용법 자체는 크게 다르지 않다.

## 1. 환경변수 로드

`.env`에 저장된 `GOOGLE_API_KEY`(및 기존 `OPENAI_API_KEY`)를 불러온다.

In [2]:
from dotenv import load_dotenv

# .env 파일의 GOOGLE_API_KEY 등 환경변수를 불러온다.
load_dotenv()

True

## 2. Gemini 모델로 기본 스트리밍 호출

`ChatGoogleGenerativeAI(model="gemini-3.6-flash")`로 모델을 만들고, `ChatOpenAI`와 동일하게 `.stream()`으로 스트리밍 호출한다. 다만 각 chunk에서 텍스트를 꺼낼 때 OpenAI 쪽 예제들에서는 `chunk.content`를 썼는데, 여기서는 `chunk.text`를 쓰고 있다. 둘 다 메시지 청크에서 실제 텍스트를 꺼내는 역할은 같다.

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

answer = llm.stream("자연어처리에 대해서 간략히 설명해줘")

# ChatOpenAI 예제에서는 chunk.content를 썼지만, 여기서는 chunk.text로 텍스트를 꺼낸다.
for chunk in answer:
    print(chunk.text, end="", flush=True)

**자연어처리(NLP, Natural Language Processing)**는 **컴퓨터가 인간의 언어(자연어)를 이해하고, 해석하고, 생성할 수 있도록 하는 인공지능(AI)의 한 분야**입니다.

여기서 '자연어'란 한국어, 영어처럼 우리가 일상에서 사용하는 말과 글을 뜻합니다. (C언어나 파이썬 같은 '프로그래밍 언어'와 구분됩니다.)

쉽게 정리하면 다음과 같습니다.

---

### 1. 주요 기능
* **텍스트 이해:** 문장의 뜻, 맥락, 감정(긍정/부정)을 파악합니다.
* **언어 변환:** 한 언어를 다른 언어로 번역합니다.
* **텍스트 생성:** 질문에 답하거나, 글을 요약하고, 새로운 문장을 만들어 냅니다.

### 2. 실생활 활용 예시
* **대화형 AI:** ChatGPT, 챗봇
* **기계 번역:** 구글 번역, 네이버 파파고
* **음성 비서:** 애플 시리(Siri), 삼성 빅스비(Bixby)
* **기타:** 스팸 메일 자동 분류, 검색엔진 연관 검색어, 텍스트 자동 요약

### 3. 왜 어려울까요?
인간의 언어는 **문맥, 비유, 중의적 표현, 신조어, 뉘앙스** 등이 복잡하게 얽혀 있어서 컴퓨터가 완벽히 이해하기 어렵습니다. 

하지만 최근 **딥러닝**과 **거대언어모델(LLM, 예: GPT)**의 발전 덕분에 AI가 사람과 거의 유사한 수준으로 언어를 다룰 수 있게 되었습니다.

## 3. 프롬프트 템플릿 + 체인으로 연결

`PromptTemplate`과 `prompt | model` 체인 구성 방식도 OpenAI 때와 완전히 동일하다. "{question}는 과일입니까?"라는 예/아니오 질문 템플릿에 "사과"를 넣어 호출해본다.

In [6]:
from langchain_core.prompts import PromptTemplate

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

prompt = PromptTemplate.from_template(
    "예/아니오 질문에 대답하세요. {question}는 과일입니까?"
)

chain = prompt | model

answer = chain.stream({"question":"사과"})

for chunk in answer:
    print(chunk.text, end="", flush=True)

예.

## 4. 안전 설정(safety_settings)

Gemini는 `safety_settings`로 성적/혐오/괴롭힘/위험 콘텐츠 등 카테고리별로 응답을 얼마나 엄격하게 차단할지 설정할 수 있다. `HarmBlockThreshold.BLOCK_NONE`으로 지정하면 해당 카테고리에 대한 차단을 하지 않는다(실습/테스트 목적으로 차단 수준을 낮출 때 사용).

> 참고: 이 셀에서 만든 `llm`은 `safety_settings`가 적용된 상태지만, 바로 다음 멀티모달 예제 셀에서는 `llm`을 다시 새로 만들면서 `safety_settings`를 지정하지 않았다. 그래서 실제 이미지 설명 호출에는 이 안전 설정이 적용되지 않은 상태다.

In [8]:
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    HarmBlockThreshold,
    HarmCategory
)

# BLOCK_NONE으로 지정한 카테고리는 해당 유형의 콘텐츠를 차단하지 않는다.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    safety_settings={
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    },
)

## 5. 멀티모달: 이미지를 보고 시 쓰기

9월 15일에 `ChatOpenAI`로 실습했던 멀티모달(이미지 입력) 패턴을 Gemini로 그대로 반복한다.

1. 로컬 이미지(`jeju-beach.jpg`)를 base64로 인코딩해서 `data:image/jpeg;base64,...` 형태의 문자열로 변환
2. `SystemMessage`로 "당신은 시인입니다"라는 역할을 부여
3. `HumanMessage`의 `content`를 리스트로 구성해서 텍스트(`type: "text"`)와 이미지(`type: "image_url"`)를 함께 전달
4. `.stream()`으로 호출하면, 모델이 이미지를 보고 시를 지어서 스트리밍으로 출력한다

In [ ]:
import base64
from langchain_core.messages import HumanMessage, SystemMessage

# 객체 생성
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

system_prompt = (
    "당신은 시인입니다. 당신의 임무는 주어진 이미지를 가지고 시를 작성하는 것입니다."
)

user_prompt = "다음의 이미지에 대한 시를 작성해주세요."

# 이미지를 base64로 인코딩
with open("jeju-beach.jpg", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

# 멀티모달 메시지 구성: content를 리스트로 만들어 텍스트와 이미지를 함께 담는다.
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(
        content=[
            {"type": "text", "text": user_prompt},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
            },
        ]
    ),
]

answer = llm.stream(messages)

for chunk in answer:
    print(chunk.text, end="", flush=True)

**옥빛 바다에 띄운 노래**

하늘 한 조각을 그대로 풀어놓았을까,
투명한 옥빛 물결이 가슴 깊이 밀려옵니다.
햇살을 듬뿍 머금은 에메랄드빛 바다는
세상의 모든 시름을 맑게 씻어내듯 반짝입니다.

검은 현무암 거친 어깨를 살포시 쓰다듬으며
밀려왔다 아득히 스러지는 하얀 파도.
바닷속 감춰진 모래알과 돌멩이조차
숨김없이 투명하게 빛나는 눈부신 세상입니다.

저 멀리 수평선에 가만히 안겨 있는 초록의 섬 하나,
수많은 세월 동안 파도의 노래를 자장가 삼아
의연히 서서 푸른 꿈을 꾸고 있습니다.

바람도 잠시 숨을 죽이는 이 거대한 평화 속에서
나 또한 한 포기 바다풀처럼 조용히 물들어 갑니다.
내 안의 어두운 마음도
이 맑고 시원한 푸르름 속에 가만히 녹아내립니다.